# Full Exploratory Data Analysis

This notebook performs a complete local EDA for the confidential classification system.

## Objectives
- connect to local DuckDB
- read `task1_clean` and `task2_clean`
- inspect schema, nulls, duplicates, and label coverage
- generate a variety of colored charts
- analyze class balance for industry and subindustry
- analyze text coverage and text length
- inspect numeric fields where relevant
- export figures to `outputs/figures`
- export a consolidated PDF to `outputs/figures/eda_report.pdf`

## Data sources
- `task1_clean`: industry modeling dataset
- `task2_clean`: subindustry modeling dataset


In [1]:
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import duckdb
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "db" / "analytics.duckdb"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

pdf_path = FIG_DIR / "eda_report.pdf"
pdf = PdfPages(pdf_path)

print("Project root:", PROJECT_ROOT)
print("DB path:", DB_PATH)
print("Figures dir:", FIG_DIR)
print("Saving PDF to:", pdf_path)


Project root: C:\Users\akash\Desktop\capstone MGT 599
DB path: C:\Users\akash\Desktop\capstone MGT 599\data\db\analytics.duckdb
Figures dir: C:\Users\akash\Desktop\capstone MGT 599\outputs\figures
Saving PDF to: C:\Users\akash\Desktop\capstone MGT 599\outputs\figures\eda_report.pdf


In [3]:
con = duckdb.connect(str(DB_PATH), read_only=True)
task1 = con.execute("SELECT * FROM task1_clean").fetchdf()
task2 = con.execute("SELECT * FROM task2_clean").fetchdf()
con.close()

print("task1_clean shape:", task1.shape)
print("task2_clean shape:", task2.shape)


IOException: IO Error: Cannot open file "\\?\C:\Users\akash\Desktop\capstone MGT 599\data\db\analytics.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\python3.11.exe (PID 8360)

## Quick preview

In [ ]:
task1.head()

In [ ]:
task2.head()

## Schema and dataset profile

In [ ]:
task1_profile = pd.DataFrame({
    "column": task1.columns,
    "dtype": [str(x) for x in task1.dtypes],
    "null_count": [task1[c].isna().sum() for c in task1.columns],
    "non_null_count": [task1[c].notna().sum() for c in task1.columns],
    "nunique": [task1[c].nunique(dropna=True) for c in task1.columns]
})
task1_profile


In [ ]:
task2_profile = pd.DataFrame({
    "column": task2.columns,
    "dtype": [str(x) for x in task2.dtypes],
    "null_count": [task2[c].isna().sum() for c in task2.columns],
    "non_null_count": [task2[c].notna().sum() for c in task2.columns],
    "nunique": [task2[c].nunique(dropna=True) for c in task2.columns]
})
task2_profile1


In [ ]:
task1_profile.to_csv(FIG_DIR / "task1_profile.csv", index=False)
task2_profile.to_csv(FIG_DIR / "task2_profile.csv", index=False)
print("Saved dataset profiles")


## Duplicate checks

In [ ]:
task1_dupes = task1.duplicated(subset=["CompanyId", "AsOfDate", "SegmentName"]).sum()
task2_dupes = task2.duplicated(subset=["CompanyId", "AsOfDate", "SegmentName"]).sum()

pd.DataFrame({
    "dataset": ["task1_clean", "task2_clean"],
    "duplicate_count": [task1_dupes, task2_dupes]
})


## Missing values

In [ ]:
task1_missing = task1.isna().sum().sort_values(ascending=False).reset_index()
task1_missing.columns = ["column", "missing_count"]
task1_missing["missing_pct"] = (task1_missing["missing_count"] / len(task1) * 100).round(2)

task2_missing = task2.isna().sum().sort_values(ascending=False).reset_index()
task2_missing.columns = ["column", "missing_count"]
task2_missing["missing_pct"] = (task2_missing["missing_count"] / len(task2) * 100).round(2)

task1_missing


In [ ]:
task2_missing

In [ ]:
task1_missing.to_csv(FIG_DIR / "task1_missing_summary.csv", index=False)
task2_missing.to_csv(FIG_DIR / "task2_missing_summary.csv", index=False)
print("Saved missing summaries")


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(task1_missing["column"], task1_missing["missing_count"], color=["#e63946", "#457b9d", "#2a9d8f", "#f4a261", "#6a4c93", "#ffb703", "#8ecae6", "#fb8500", "#90be6d", "#577590"][:len(task1_missing)])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Missing Count")
plt.title("Missing Values by Column - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_missing_values_bar.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(task2_missing["column"], task2_missing["missing_count"], color=["#8ecae6", "#219ebc", "#023047", "#ffb703", "#fb8500"][:len(task2_missing)])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Missing Count")
plt.title("Missing Values by Column - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_missing_values_bar.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
nonzero_missing_task1 = task1_missing[task1_missing["missing_count"] > 0]
if not nonzero_missing_task1.empty:
    plt.figure(figsize=(7, 7))
    plt.pie(
        nonzero_missing_task1["missing_count"],
        labels=nonzero_missing_task1["column"],
        autopct="%1.1f%%",
        colors=["#e76f51", "#2a9d8f", "#264653", "#e9c46a", "#f4a261"]
    )
    plt.title("Share of Missing Values - task1_clean")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "task1_missing_values_pie.png", dpi=200, bbox_inches="tight")
    pdf.savefig()
    plt.show()
else:
    print("No missing values to plot for task1_clean pie chart")


## Label distributions

In [ ]:
industry_counts = task1["MstarGlobal"].fillna("MISSING").value_counts().reset_index()
industry_counts.columns = ["MstarGlobal", "count"]
industry_counts["pct"] = (industry_counts["count"] / len(task1) * 100).round(2)

subindustry_counts = task2["Subindustry"].fillna("MISSING").value_counts().reset_index()
subindustry_counts.columns = ["Subindustry", "count"]
subindustry_counts["pct"] = (subindustry_counts["count"] / len(task2) * 100).round(2)

industry_counts.head(15)


In [ ]:
subindustry_counts.head(15)

In [ ]:
industry_counts.to_csv(FIG_DIR / "industry_distribution.csv", index=False)
subindustry_counts.to_csv(FIG_DIR / "subindustry_distribution.csv", index=False)
print("Saved label distributions")


In [ ]:
top_n = 20
plot_df = industry_counts.head(top_n).sort_values("count")

plt.figure(figsize=(10, 8))
plt.barh(plot_df["MstarGlobal"], plot_df["count"], color="#4c78a8")
plt.xlabel("Count")
plt.ylabel("Industry")
plt.title(f"Top {top_n} Industry Classes")
plt.tight_layout()
plt.savefig(FIG_DIR / "industry_class_distribution_barh.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
top_n = 12
plot_df = industry_counts.head(top_n)

plt.figure(figsize=(10, 5))
plt.bar(plot_df["MstarGlobal"], plot_df["count"], color="#f28e2b")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Count")
plt.title(f"Top {top_n} Industry Classes")
plt.tight_layout()
plt.savefig(FIG_DIR / "industry_class_distribution_bar.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
top_n = 25
plot_df = subindustry_counts.head(top_n).sort_values("count")

plt.figure(figsize=(11, 10))
plt.barh(plot_df["Subindustry"], plot_df["count"], color="#59a14f")
plt.xlabel("Count")
plt.ylabel("Subindustry")
plt.title(f"Top {top_n} Subindustry Classes")
plt.tight_layout()
plt.savefig(FIG_DIR / "subindustry_class_distribution_barh.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
top_n = 10
plot_df = subindustry_counts.head(top_n)

plt.figure(figsize=(8, 8))
plt.pie(
    plot_df["count"],
    labels=plot_df["Subindustry"],
    autopct="%1.1f%%",
    colors=["#577590", "#43aa8b", "#90be6d", "#f9c74f", "#f8961e", "#f3722c", "#f94144", "#277da1", "#4d908e", "#7b2cbf"]
)
plt.title(f"Top {top_n} Subindustry Share")
plt.tight_layout()
plt.savefig(FIG_DIR / "subindustry_top10_pie.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


## label analysis

In [ ]:
industry_rare = industry_counts[industry_counts["count"] <= 5].copy()
subindustry_rare = subindustry_counts[subindustry_counts["count"] <= 5].copy()

pd.DataFrame({
    "dataset": ["industry", "subindustry"],
    "rare_labels_le_5": [len(industry_rare), len(subindustry_rare)]
})


In [ ]:
tail_industry = industry_counts.tail(20).sort_values("count")

plt.figure(figsize=(10, 8))
plt.barh(tail_industry["MstarGlobal"], tail_industry["count"], color="#b56576")
plt.xlabel("Count")
plt.ylabel("Industry")
plt.title("Rarest 20 Industry Classes")
plt.tight_layout()
plt.savefig(FIG_DIR / "industry_rarest_classes.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
tail_sub = subindustry_counts.tail(20).sort_values("count")

plt.figure(figsize=(11, 9))
plt.barh(tail_sub["Subindustry"], tail_sub["count"], color="#6d597a")
plt.xlabel("Count")
plt.ylabel("Subindustry")
plt.title("Rarest 20 Subindustry Classes")
plt.tight_layout()
plt.savefig(FIG_DIR / "subindustry_rarest_classes.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


## Text feature construction for EDA

In [ ]:
task1["LongProfile_len"] = task1["LongProfile"].fillna("").astype(str).str.len()
task1["SegmentDescription_len"] = task1["SegmentDescription"].fillna("").astype(str).str.len()
task1["SegmentName_len"] = task1["SegmentName"].fillna("").astype(str).str.len()

task2["SegmentDescription_len"] = task2["SegmentDescription"].fillna("").astype(str).str.len()
task2["SegmentName_len"] = task2["SegmentName"].fillna("").astype(str).str.len()

task1["combined_text"] = (
    task1["LongProfile"].fillna("").astype(str) + " " +
    task1["SegmentDescription"].fillna("").astype(str) + " " +
    task1["SegmentName"].fillna("").astype(str)
).str.strip()

task2["combined_text"] = (
    task2["SegmentDescription"].fillna("").astype(str) + " " +
    task2["SegmentName"].fillna("").astype(str)
).str.strip()

task1["combined_text_len"] = task1["combined_text"].str.len()
task2["combined_text_len"] = task2["combined_text"].str.len()

print("Text features added")


## Text coverage

In [ ]:
coverage_task1 = pd.DataFrame({
    "column": ["LongProfile", "SegmentDescription", "SegmentName", "MstarGlobal"],
    "non_null_count": [
        task1["LongProfile"].notna().sum(),
        task1["SegmentDescription"].notna().sum(),
        task1["SegmentName"].notna().sum(),
        task1["MstarGlobal"].notna().sum(),
    ],
    "total_rows": len(task1)
})
coverage_task1["coverage_pct"] = (coverage_task1["non_null_count"] / coverage_task1["total_rows"] * 100).round(2)
coverage_task1


In [ ]:
coverage_task2 = pd.DataFrame({
    "column": ["SegmentDescription", "SegmentName", "Subindustry"],
    "non_null_count": [
        task2["SegmentDescription"].notna().sum(),
        task2["SegmentName"].notna().sum(),
        task2["Subindustry"].notna().sum(),
    ],
    "total_rows": len(task2)
})
coverage_task2["coverage_pct"] = (coverage_task2["non_null_count"] / coverage_task2["total_rows"] * 100).round(2)
coverage_task2


In [ ]:
coverage_task1.to_csv(FIG_DIR / "coverage_task1.csv", index=False)
coverage_task2.to_csv(FIG_DIR / "coverage_task2.csv", index=False)

plt.figure(figsize=(8, 5))
plt.bar(coverage_task1["column"], coverage_task1["coverage_pct"], color=["#264653", "#2a9d8f", "#e9c46a", "#f4a261"])
plt.ylim(0, 105)
plt.ylabel("Coverage %")
plt.title("Field Coverage - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_coverage_bar.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(coverage_task2["column"], coverage_task2["coverage_pct"], color=["#3a86ff", "#8338ec", "#ff006e"])
plt.ylim(0, 105)
plt.ylabel("Coverage %")
plt.title("Field Coverage - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_coverage_bar.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


## Text length distributions

In [ ]:
task1_length_summary = task1[["LongProfile_len", "SegmentDescription_len", "SegmentName_len", "combined_text_len"]].describe().T.reset_index()
task1_length_summary.columns = ["feature", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]

task2_length_summary = task2[["SegmentDescription_len", "SegmentName_len", "combined_text_len"]].describe().T.reset_index()
task2_length_summary.columns = ["feature", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]

task1_length_summary


In [ ]:
task2_length_summary

In [ ]:
task1_length_summary.to_csv(FIG_DIR / "task1_text_length_summary.csv", index=False)
task2_length_summary.to_csv(FIG_DIR / "task2_text_length_summary.csv", index=False)
print("Saved text length summaries")


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task1["LongProfile_len"], bins=50, color="#457b9d", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("LongProfile Length Distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "longprofile_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task1["SegmentDescription_len"], bins=50, color="#e76f51", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("SegmentDescription Length Distribution - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_segmentdescription_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task1["SegmentName_len"], bins=50, color="#2a9d8f", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("SegmentName Length Distribution - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_segmentname_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task1["combined_text_len"], bins=60, color="#6a4c93", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("Combined Text Length Distribution - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_combined_text_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task2["SegmentDescription_len"], bins=50, color="#f4a261", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("SegmentDescription Length Distribution - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_segmentdescription_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task2["SegmentName_len"], bins=50, color="#90be6d", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("SegmentName Length Distribution - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_segmentname_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(task2["combined_text_len"], bins=60, color="#f94144", edgecolor="black")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("Combined Text Length Distribution - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_combined_text_length_hist.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
boxplot_data = [
    task1["LongProfile_len"],
    task1["SegmentDescription_len"],
    task1["SegmentName_len"],
    task1["combined_text_len"]
]

plt.figure(figsize=(10, 6))
bp = plt.boxplot(boxplot_data, patch_artist=True, labels=["LongProfile", "SegmentDescription", "SegmentName", "CombinedText"])
colors = ["#577590", "#43aa8b", "#f9c74f", "#f3722c"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
plt.ylabel("Character Length")
plt.title("Text Length Comparison - task1_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task1_text_length_boxplot.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
boxplot_data = [
    task2["SegmentDescription_len"],
    task2["SegmentName_len"],
    task2["combined_text_len"]
]

plt.figure(figsize=(9, 6))
bp = plt.boxplot(boxplot_data, patch_artist=True, labels=["SegmentDescription", "SegmentName", "CombinedText"])
colors = ["#3a86ff", "#8338ec", "#ff006e"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
plt.ylabel("Character Length")
plt.title("Text Length Comparison - task2_clean")
plt.tight_layout()
plt.savefig(FIG_DIR / "task2_text_length_boxplot.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


## Numeric feature checks from task1_clean

In [ ]:
numeric_cols_task1 = []
for col in ["Revenue", "total_revenue_company_as_of", "revenue_share"]:
    if col in task1.columns:
        numeric_cols_task1.append(col)

numeric_cols_task1


In [ ]:
if numeric_cols_task1:
    task1[numeric_cols_task1].describe().T
else:
    print("No numeric columns found")


In [ ]:
if "Revenue" in task1.columns:
    plt.figure(figsize=(9, 5))
    plt.hist(task1["Revenue"].dropna(), bins=50, color="#118ab2", edgecolor="black")
    plt.xlabel("Revenue")
    plt.ylabel("Frequency")
    plt.title("Revenue Distribution - task1_clean")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "task1_revenue_hist.png", dpi=200, bbox_inches="tight")
    pdf.savefig()
    plt.show()


In [ ]:
if "revenue_share" in task1.columns:
    plt.figure(figsize=(9, 5))
    plt.hist(task1["revenue_share"].dropna(), bins=50, color="#06d6a0", edgecolor="black")
    plt.xlabel("Revenue Share")
    plt.ylabel("Frequency")
    plt.title("Revenue Share Distribution - task1_clean")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "task1_revenue_share_hist.png", dpi=200, bbox_inches="tight")
    pdf.savefig()
    plt.show()


## Grouped analysis by label

In [ ]:
industry_text_stats = task1.groupby("MstarGlobal").agg(
    row_count=("MstarGlobal", "size"),
    avg_longprofile_len=("LongProfile_len", "mean"),
    avg_segmentdescription_len=("SegmentDescription_len", "mean"),
    avg_segmentname_len=("SegmentName_len", "mean"),
    avg_combined_len=("combined_text_len", "mean")
).reset_index().sort_values("row_count", ascending=False)

industry_text_stats.head(15)


In [ ]:
subindustry_text_stats = task2.groupby("Subindustry").agg(
    row_count=("Subindustry", "size"),
    avg_segmentdescription_len=("SegmentDescription_len", "mean"),
    avg_segmentname_len=("SegmentName_len", "mean"),
    avg_combined_len=("combined_text_len", "mean")
).reset_index().sort_values("row_count", ascending=False)

subindustry_text_stats.head(15)


In [ ]:
industry_text_stats.to_csv(FIG_DIR / "industry_text_stats.csv", index=False)
subindustry_text_stats.to_csv(FIG_DIR / "subindustry_text_stats.csv", index=False)
print("Saved grouped text stats")


In [ ]:
plot_df = industry_text_stats.head(15).sort_values("avg_combined_len")

plt.figure(figsize=(10, 7))
plt.barh(plot_df["MstarGlobal"], plot_df["avg_combined_len"], color="#ef476f")
plt.xlabel("Average Combined Text Length")
plt.ylabel("Industry")
plt.title("Top 15 Industries by Average Combined Text Length")
plt.tight_layout()
plt.savefig(FIG_DIR / "industry_avg_combined_text_length.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


In [ ]:
plot_df = subindustry_text_stats.head(15).sort_values("avg_combined_len")

plt.figure(figsize=(11, 8))
plt.barh(plot_df["Subindustry"], plot_df["avg_combined_len"], color="#ffd166")
plt.xlabel("Average Combined Text Length")
plt.ylabel("Subindustry")
plt.title("Top 15 Subindustries by Average Combined Text Length")
plt.tight_layout()
plt.savefig(FIG_DIR / "subindustry_avg_combined_text_length.png", dpi=200, bbox_inches="tight")
pdf.savefig()
plt.show()


## Modeling readiness summary

In [ ]:
model_readiness = pd.DataFrame({
    "check": [
        "task1 rows",
        "task2 rows",
        "task1 missing SegmentDescription",
        "task2 missing SegmentDescription",
        "unique industries",
        "unique subindustries",
        "task1 avg combined text length",
        "task2 avg combined text length",
        "task1 duplicate key rows",
        "task2 duplicate key rows"
    ],
    "value": [
        len(task1),
        len(task2),
        int(task1["SegmentDescription"].isna().sum()),
        int(task2["SegmentDescription"].isna().sum()),
        int(task1["MstarGlobal"].nunique(dropna=True)),
        int(task2["Subindustry"].nunique(dropna=True)),
        round(task1["combined_text_len"].mean(), 2),
        round(task2["combined_text_len"].mean(), 2),
        int(task1_dupes),
        int(task2_dupes),
    ]
})
model_readiness


In [ ]:
model_readiness.to_csv(FIG_DIR / "model_readiness_summary.csv", index=False)
print("Saved model readiness summary")


## Final summary page for PDF

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

summary_text = f'''
EDA SUMMARY

Dataset shapes
- task1_clean rows: {len(task1)}
- task2_clean rows: {len(task2)}

Label space
- unique industries: {task1["MstarGlobal"].nunique(dropna=True)}
- unique subindustries: {task2["Subindustry"].nunique(dropna=True)}

Missingness
- task1 SegmentDescription missing: {task1["SegmentDescription"].isna().sum()}
- task2 SegmentDescription missing: {task2["SegmentDescription"].isna().sum()}

Text readiness
- task1 avg combined text length: {round(task1["combined_text_len"].mean(), 2)}
- task2 avg combined text length: {round(task2["combined_text_len"].mean(), 2)}

Data quality
- task1 duplicate key rows: {task1_dupes}
- task2 duplicate key rows: {task2_dupes}

Recommendations from my eda
- to use combined text fields for baseline TF-IDF modeling
- to drop AsOfDate for modeling since it is constant
- to watch for class imbalance for both industry and subindustry
- to consider rare-label handling for sparse classes
'''

ax.text(0.03, 0.97, summary_text, va="top", fontsize=13)
ax.axis("off")
pdf.savefig()
plt.close()


## Close connections and finalize report

In [ ]:
pdf.close()
con.close()
print("EDA PDF generated successfully")
print("PDF path:", pdf_path)
